# 05 — Local deployment test

This notebook reloads only the standalone bundle from notebook 4, then repeats validation, feature derivation, OOD detection, matrix construction, scoring, and append-only persistence explicitly. It does not call `firmaware.predict`.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import json
import time

import joblib
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODEL_DIR = ROOT / "notebook_runs" / "model"
required = [MODEL_DIR / "model.joblib", MODEL_DIR / "preprocessor.joblib", MODEL_DIR / "metadata.json"]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Run notebook 04 first. Missing: {missing}")

model = joblib.load(MODEL_DIR / "model.joblib")
preprocessor = joblib.load(MODEL_DIR / "preprocessor.joblib")
metadata = json.loads((MODEL_DIR / "metadata.json").read_text(encoding="utf-8"))
print(f"loaded model={metadata['model_name']} threshold={metadata['threshold']:.2f} run={metadata['timestamp']}")

loaded model=xgboost threshold=0.04 run=2026-07-23T12:51:24.559379Z


## Validate the scoring contract explicitly

In [2]:
IDS = ["deployment_id", "device_id", "site_id", "firmware_fingerprint"]
CATEGORICAL = ["vendor_name", "device_type", "hardware_series", "fleet_tier", "site_criticality", "deployment_type"]
NUMERIC = ["version_jump_magnitude", "kernel_touched", "bootloader_touched", "protocol_mismatch_flag", "maintenance_window", "network_stress_score", "error_rate_predeploy", "uptime_days", "past_failure_count", "firmware_release_age_days", "cve_count", "max_cvss_score", "cross_vendor_dependency_count", "dependent_device_count"]
DERIVED = ["major_version_jump", "major_version_changed", "emergency_no_maintenance", "core_system_touched"]
DATE = "deployment_date"
EXPECTED = IDS + CATEGORICAL + ["current_firmware", "target_firmware"] + NUMERIC + [DATE]

upcoming_path = ROOT / "data" / "upcoming_deployments.csv"
upcoming = pd.read_csv(upcoming_path)
missing_columns = sorted(set(EXPECTED) - set(upcoming.columns))
assert not missing_columns, f"Missing scoring columns: {missing_columns}"
assert not upcoming["deployment_id"].duplicated().any()
for column in NUMERIC:
    before = upcoming[column].notna()
    upcoming[column] = pd.to_numeric(upcoming[column], errors="coerce")
    assert not (before & upcoming[column].isna()).any(), column
print(f"validated rows={len(upcoming)} columns={len(upcoming.columns)}")

validated rows=55 columns=27


## Derive and transform with persisted state

In [3]:
def major(value):
    try:
        return int(str(value).split(".", 1)[0])
    except (TypeError, ValueError):
        return np.nan

scoring = upcoming.copy()
current_major = scoring["current_firmware"].map(major)
target_major = scoring["target_firmware"].map(major)
scoring["major_version_jump"] = target_major - current_major
scoring["major_version_changed"] = np.where(scoring["major_version_jump"].isna(), np.nan, scoring["major_version_jump"].ne(0).astype(int))
scoring["emergency_no_maintenance"] = (scoring["deployment_type"].eq("EMERGENCY") & scoring["maintenance_window"].eq(0)).astype(int)
scoring["core_system_touched"] = (scoring["kernel_touched"].fillna(0).ne(0) | scoring["bootloader_touched"].fillna(0).ne(0)).astype(int)
deployment_ids = scoring["deployment_id"].copy()
scoring = scoring.drop(columns=IDS + ["current_firmware", "target_firmware", DATE])
scoring.index = pd.Index(deployment_ids, name="deployment_id")

encoder = preprocessor["encoder"]
numeric_columns = preprocessor["numeric_columns"]
categorical_columns = preprocessor["categorical_columns"]
feature_list = preprocessor["feature_list"]
medians = pd.Series(preprocessor["medians"])

unseen_rows = []
for deployment_id, row in scoring[categorical_columns].iterrows():
    unseen = {}
    for index, column in enumerate(categorical_columns):
        if row[column] not in set(encoder.categories_[index]):
            unseen[column] = row[column]
    unseen_rows.append({"deployment_id": deployment_id, "unseen": unseen})
ood = pd.DataFrame(unseen_rows)

numeric = scoring[numeric_columns].astype(float).fillna(medians)
encoded_values = encoder.transform(scoring[categorical_columns])
encoded_names = encoder.get_feature_names_out(categorical_columns).tolist()
encoded = pd.DataFrame(encoded_values, columns=encoded_names, index=scoring.index)
matrix = pd.concat([numeric, encoded], axis=1).reindex(columns=feature_list, fill_value=0.0)
matrix.loc[:, numeric_columns] = preprocessor["scaler"].transform(matrix[numeric_columns])
matrix = matrix.astype(float)

assert matrix.columns.tolist() == feature_list
print(f"matrix={matrix.shape}; OOD rows={int(ood['unseen'].map(bool).sum())}")
display(ood.loc[ood["unseen"].map(bool)])

matrix=(55, 85); OOD rows=1


,deployment_id,unseen
54,UPC_00055,"{'vendor_name': 'Moxa', 'device_type': 'EdgeAI..."


## Score using one persisted decision line

In [4]:
probabilities = model.predict_proba(matrix.to_numpy())[:, 1]
threshold = float(metadata["threshold"])
scores = pd.DataFrame({
    "deployment_id": ood["deployment_id"],
    "risk_probability": [round(float(value), 4) for value in probabilities],
    "risk_prediction": ["NO_GO" if value >= threshold else "GO" for value in probabilities],
    "risk_band": ["HIGH" if value >= threshold else "MEDIUM" if value >= threshold / 2 else "LOW" for value in probabilities],
    "unseen_categories": [json.dumps(value, sort_keys=True, separators=(",", ":")) for value in ood["unseen"]],
    "model_run": metadata["timestamp"],
})
display(scores.head())
display(scores[["risk_prediction", "risk_band"]].value_counts().to_frame("rows"))

,deployment_id,risk_probability,risk_prediction,risk_band,unseen_categories,model_run
0,UPC_00001,0.2078,NO_GO,HIGH,{},2026-07-23T12:51:24.559379Z
1,UPC_00002,0.2337,NO_GO,HIGH,{},2026-07-23T12:51:24.559379Z
2,UPC_00003,0.2530,NO_GO,HIGH,{},2026-07-23T12:51:24.559379Z
3,UPC_00004,0.2402,NO_GO,HIGH,{},2026-07-23T12:51:24.559379Z
4,UPC_00005,0.3671,NO_GO,HIGH,{},2026-07-23T12:51:24.559379Z


,,rows
risk_prediction,risk_band,
NO_GO,HIGH,55


## Append-only local deployment behavior

Two invocations append two complete batches. The input contains one OOD row (Moxa with new device/hardware categories), and that row remains scoreable.

In [5]:
deployment_dir = ROOT / "notebook_runs" / "deployment"
deployment_dir.mkdir(parents=True, exist_ok=True)
output_path = deployment_dir / "scores.csv"
if output_path.exists():
    output_path.unlink()

def append_scores(batch, path):
    scored = batch.copy()
    scored["scored_at"] = datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")
    if path.exists() and path.stat().st_size:
        existing_columns = pd.read_csv(path, nrows=0).columns.tolist()
        assert existing_columns == scored.columns.tolist()
        scored.to_csv(path, mode="a", header=False, index=False, float_format="%.4f")
    else:
        scored.to_csv(path, mode="a", header=True, index=False, float_format="%.4f")
    return scored

first = append_scores(scores, output_path)
time.sleep(0.002)
second = append_scores(scores, output_path)
history = pd.read_csv(output_path)

assert len(history) == len(upcoming) * 2
assert history["scored_at"].nunique() == 2
assert int(first["unseen_categories"].ne("{}").sum()) == 1
assert first.loc[first["unseen_categories"].ne("{}"), "unseen_categories"].str.contains("Moxa").all()
assert set(first["model_run"]) == {metadata["timestamp"]}
print(f"PASS: {len(history)} append-only rows, {history['scored_at'].nunique()} scoring invocations")
print(output_path)

PASS: 110 append-only rows, 2 scoring invocations


C:\Users\pmkul\Downloads\FirmAware\notebook_runs\deployment\scores.csv


## Smoke fixture

The committed five-row deployment fixture is separately contract-shaped and contains exactly one Moxa row for cloud smoke testing.

In [6]:
smoke = pd.read_csv(ROOT / "deploy" / "fixtures" / "upcoming_smoke.csv")
assert len(smoke) == 5
assert set(EXPECTED).issubset(smoke.columns)
assert int(smoke["vendor_name"].eq("Moxa").sum()) == 1
print("PASS: five-row local deployment fixture with one deliberate OOD vendor")

PASS: five-row local deployment fixture with one deliberate OOD vendor
